In [ ]:
import pandas as pd

data = pd.read_csv("flight_data_cleaned.csv")
data["Scheduled_DateTime"] = pd.to_datetime(data["Scheduled_DateTime"])

weather_cols = [
    c for c in data.columns
    if c.startswith("weather__hourly__")
]

for (airport, date), group in data.groupby(
    ["From", data["Scheduled_DateTime"].dt.date]
):
    group = group.sort_values("Scheduled_DateTime")

    for _, flight in group.iterrows():
        matches = group[
            (group["Scheduled_DateTime"] >= flight["Scheduled_DateTime"]) &
            (group["Scheduled_DateTime"] <= flight["Scheduled_DateTime"] + pd.Timedelta(hours=3))
        ]

        matches = matches[
            matches[weather_cols].eq(flight[weather_cols]).all(axis=1)
        ]

        if len(matches) >= 3:
            span = (
                matches["Scheduled_DateTime"].max() -
                matches["Scheduled_DateTime"].min()
            )

            if span >= pd.Timedelta(hours=2):
                print("Airport:", airport, "Date:", date)
                print(matches[["Scheduled_DateTime"] + weather_cols])
                break

In [1]:
import pandas as pd
import numpy as np

# Load original cleaned dataset
data = pd.read_csv("flight_data_cleaned.csv")

# Create independent copy
df = data.copy(deep=True)

# Verify
print(df.shape)
df.head()

(10642, 39)


,Used Date,From,To,Airline,Scheduled Departure,SDEP,DEP,SARR,ARR,Departure Delay,...,Past_windspeedKmph,Past_precipMM,Past_humidity,Past_pressure,Past_cloudcover,Past_weatherDesc,Hourly_Airport_Departures,Airport_Congestion_Index,Is_Weekend,Is_Delayed
0,2019-01-02,BOM,DEL,Vistara,01:10,110,130,320,325,20,...,7.3,0.0,62,1017.1,87,3,7.0,0.040698,0,1
1,2019-01-09,DEL,HYD,Spicejet,22:05,2205,40,25,252,155,...,9.4,0.0,67,1017.7,0,0,56.0,0.225806,0,1
2,2019-01-22,CCU,DEL,SpiceJet,18:30,1830,1838,2055,2106,8,...,5.2,0.0,61,1014.4,0,0,90.0,1.000000,0,1
3,2019-01-23,CCU,DEL,SpiceJet,18:30,1830,1830,2055,2100,0,...,3.6,0.0,55,1014.0,0,0,90.0,1.000000,0,0
4,2019-01-24,CCU,DEL,SpiceJet,18:30,1830,1830,2055,2101,0,...,4.3,0.0,58,1014.7,1,0,90.0,1.000000,0,0


In [2]:
print("Departure airports:")
print(df["From"].value_counts())

print("\nFlight date range:")
dates = pd.to_datetime(df["Used Date"], errors="coerce")
print(dates.min())
print(dates.max())

print("\nRelevant columns:")
print([c for c in df.columns
       if "date" in c.lower()
       or "time" in c.lower()
       or "sched" in c.lower()])

Departure airports:
From
BLR    5393
CCU    2181
DEL    2041
BOM    1027
Name: count, dtype: int64

Flight date range:
2019-01-02 00:00:00
2020-01-31 00:00:00

Relevant columns:
['Used Date', 'Scheduled Departure', 'Scheduled_DateTime', 'time']


In [3]:
import requests
from io import StringIO

API_KEY = "9MR7M2PVEJNEX3F9MXKF294PU"

location = "Delhi,India"  # Replace with airport coordinates
start_date = "2019-01-01"  # Replace with dataset start
end_date = "2019-01-07"    # Test with one week first

url = (
    "https://weather.visualcrossing.com/"
    "VisualCrossingWebServices/rest/services/timeline/"
    f"{location}/{start_date}/{end_date}"
)

params = {
    "key": API_KEY,
    "unitGroup": "metric",
    "include": "hours",
    "contentType": "csv"
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

weather = pd.read_csv(StringIO(response.text))
print(weather.shape)
weather.head()

(168, 24)


,name,datetime,temp,feelslike,dew,humidity,precip,precipprob,preciptype,snow,...,sealevelpressure,cloudcover,visibility,solarradiation,solarenergy,uvindex,severerisk,conditions,icon,stations
0,"Delhi, DL, India",2019-01-01T00:00:00,9.0,9.0,6.0,81.49,0.0,0,NaN,0,...,1020.0,0.0,1.0,0.0,0.0,0,NaN,Clear,fog,"VIDP,42181099999"
1,"Delhi, DL, India",2019-01-01T01:00:00,8.6,7.4,7.1,90.29,0.0,0,NaN,0,...,1021.0,0.0,NaN,0.0,0.0,0,NaN,Clear,clear-night,NaN
2,"Delhi, DL, India",2019-01-01T02:00:00,6.9,6.9,6.7,98.17,0.0,0,NaN,0,...,1021.3,0.0,1.0,0.0,0.0,0,NaN,Clear,fog,"42182099999,42181099999"
3,"Delhi, DL, India",2019-01-01T03:00:00,8.1,6.9,6.8,91.50,0.0,0,NaN,0,...,1021.0,0.0,NaN,0.0,0.0,0,NaN,Clear,clear-night,NaN
4,"Delhi, DL, India",2019-01-01T04:00:00,6.1,4.7,5.2,93.95,0.0,0,NaN,0,...,1021.0,0.0,NaN,0.0,0.0,0,NaN,Clear,clear-night,NaN


In [4]:
weather_cols = [
    "datetime",
    "windspeed",
    "precip",
    "humidity",
    "visibility",
    "sealevelpressure",
    "cloudcover",
    "conditions"
]

weather = weather[weather_cols].copy()

weather.head()

,datetime,windspeed,precip,humidity,visibility,sealevelpressure,cloudcover,conditions
0,2019-01-01T00:00:00,0.0,0.0,81.49,1.0,1020.0,0.0,Clear
1,2019-01-01T01:00:00,7.6,0.0,90.29,NaN,1021.0,0.0,Clear
2,2019-01-01T02:00:00,0.0,0.0,98.17,1.0,1021.3,0.0,Clear
3,2019-01-01T03:00:00,7.2,0.0,91.50,NaN,1021.0,0.0,Clear
4,2019-01-01T04:00:00,6.8,0.0,93.95,NaN,1021.0,0.0,Clear


In [5]:
print("Departure airports:")
print(df["From"].value_counts())

print("\nDate range:")
dates = pd.to_datetime(df["Used Date"], errors="coerce")
print("Start:", dates.min())
print("End:", dates.max())

print("\nScheduled departure columns:")
print([c for c in df.columns if "sched" in c.lower()])

Departure airports:
From
BLR    5393
CCU    2181
DEL    2041
BOM    1027
Name: count, dtype: int64

Date range:
Start: 2019-01-02 00:00:00
End: 2020-01-31 00:00:00

Scheduled departure columns:
['Scheduled Departure', 'Scheduled_DateTime']


In [6]:
import pandas as pd
import requests
from io import StringIO
from pathlib import Path

# Uses your existing API_KEY variable
airports = {
    "BLR": "12.9716,77.5946",
    "CCU": "22.6547,88.4467",
    "DEL": "28.5562,77.1000",
    "BOM": "19.0896,72.8656"
}

start_date = pd.Timestamp("2019-01-02")
end_date = pd.Timestamp("2020-01-31")

elements = [
    "datetime",
    "windspeed",
    "precip",
    "humidity",
    "visibility",
    "pressure",
    "cloudcover",
    "conditions"
]

output_dir = Path("airport_weather")
output_dir.mkdir(exist_ok=True)

# Generate monthly download intervals
months = pd.period_range(start_date, end_date, freq="M")

def download_airport_weather(airport, location):
    frames = []

    for month in months:
        start = max(month.start_time.normalize(), start_date)
        end = min(month.end_time.normalize(), end_date)

        url = (
            "https://weather.visualcrossing.com/"
            "VisualCrossingWebServices/rest/services/timeline/"
            f"{location}/{start:%Y-%m-%d}/{end:%Y-%m-%d}"
        )

        params = {
            "key": API_KEY,
            "unitGroup": "metric",
            "include": "hours",
            "contentType": "csv",
            "elements": ",".join(elements)
        }

        response = requests.get(url, params=params, timeout=120)

        if not response.ok:
            raise RuntimeError(
                f"{airport} {month}: HTTP {response.status_code}: "
                f"{response.text[:300]}"
            )

        chunk = pd.read_csv(StringIO(response.text))
        frames.append(chunk)

        print(f"{airport} {month}: {len(chunk)} rows")

    result = pd.concat(frames, ignore_index=True)
    result["From"] = airport

    result.to_csv(
        output_dir / f"{airport}_weather.csv",
        index=False
    )

    return result

In [7]:
# Temporarily test only January 2019
months = pd.period_range("2019-01", "2019-01", freq="M")

test_weather = download_airport_weather(
    "BLR", airports["BLR"]
)

print(test_weather.shape)
display(test_weather.head())

BLR 2019-01: 720 rows
(720, 9)


,datetime,humidity,precip,windspeed,sealevelpressure,cloudcover,visibility,conditions,From
0,2019-01-02T00:00:00,76.76,0.0,3.7,1021.0,0.0,6.0,Clear,BLR
1,2019-01-02T01:00:00,76.68,0.0,0.9,1021.0,0.0,6.0,Clear,BLR
2,2019-01-02T02:00:00,76.05,0.0,3.6,1020.0,10.0,4.0,Clear,BLR
3,2019-01-02T03:00:00,87.70,0.0,2.7,1020.0,0.0,4.0,Clear,BLR
4,2019-01-02T04:00:00,100.00,0.0,1.8,1020.0,59.4,2.0,Partially cloudy,BLR


In [8]:
flight_hours = df[["From", "Scheduled_DateTime"]].copy()

flight_hours["Scheduled_DateTime"] = pd.to_datetime(
    flight_hours["Scheduled_DateTime"],
    errors="coerce"
)

flight_hours["Hour"] = flight_hours[
    "Scheduled_DateTime"
].dt.floor("h")

unique_hours = flight_hours[
    ["From", "Hour"]
].dropna().drop_duplicates()

print("Flights:", len(df))
print("Unique airport-hours:", len(unique_hours))
print(unique_hours.groupby("From").size())

Flights: 10642
Unique airport-hours: 4946
From
BLR    1765
BOM     553
CCU    1332
DEL    1296
dtype: int64


In [9]:
unique_days = unique_hours.copy()

unique_days["Date"] = unique_days["Hour"].dt.date

airport_days = unique_days[
    ["From", "Date"]
].drop_duplicates()

print("Unique airport-days:", len(airport_days))
print(airport_days.groupby("From").size())

Unique airport-days: 604
From
BLR    180
BOM    160
CCU    129
DEL    135
dtype: int64


In [10]:
print("Airport-days:", len(airport_days))
print("Maximum hourly records:", len(airport_days) * 24)

Airport-days: 604
Maximum hourly records: 14496


In [11]:
airport = "BLR"
location = airports[airport]
date = str(
    airport_days.loc[
        airport_days["From"] == airport, "Date"
    ].iloc[0]
)

url = (
    "https://weather.visualcrossing.com/"
    "VisualCrossingWebServices/rest/services/timeline/"
    f"{location}/{date}/{date}"
)

params = {
    "key": API_KEY,
    "unitGroup": "metric",
    "include": "hours",
    "contentType": "csv",
    "elements": ",".join(elements)
}

response = requests.get(url, params=params, timeout=60)

print("Status:", response.status_code)

if response.ok:
    test = pd.read_csv(StringIO(response.text))
    print("Shape:", test.shape)
    print("Columns:", test.columns.tolist())
    display(test.head())
else:
    print(response.text[:500])

Status: 200
Shape: (24, 8)
Columns: ['datetime', 'humidity', 'precip', 'windspeed', 'sealevelpressure', 'cloudcover', 'visibility', 'conditions']


,datetime,humidity,precip,windspeed,sealevelpressure,cloudcover,visibility,conditions
0,2019-01-31T00:00:00,93.88,0,8.4,1020.0,0.0,6.0,Clear
1,2019-01-31T01:00:00,100.00,0,5.5,1019.0,12.7,6.0,Clear
2,2019-01-31T02:00:00,89.29,0,7.5,1019.0,30.0,4.0,Partially cloudy
3,2019-01-31T03:00:00,100.00,0,14.7,1019.0,50.0,4.0,Partially cloudy
4,2019-01-31T04:00:00,100.00,0,13.0,1018.0,50.0,4.0,Partially cloudy


In [12]:
from pathlib import Path

print(list(Path("airport_weather").glob("*.csv")))

[WindowsPath('airport_weather/BLR_weather.csv')]


In [13]:
import pandas as pd
from pathlib import Path

files = list(Path("airport_weather").glob("*_weather.csv"))

if files:
    weather_all = pd.concat(
        [pd.read_csv(file) for file in files],
        ignore_index=True
    )

    print(weather_all.shape)
    print(weather_all["From"].value_counts())
else:
    print("No saved airport weather files found.")

(720, 9)
From
BLR    720
Name: count, dtype: int64


In [14]:
print(weather_all.columns.tolist())
print(weather_all.head(3))

['datetime', 'humidity', 'precip', 'windspeed', 'sealevelpressure', 'cloudcover', 'visibility', 'conditions', 'From']
              datetime  humidity  precip  windspeed  sealevelpressure  \
0  2019-01-02T00:00:00     76.76     0.0        3.7            1021.0   
1  2019-01-02T01:00:00     76.68     0.0        0.9            1021.0   
2  2019-01-02T02:00:00     76.05     0.0        3.6            1020.0   

   cloudcover  visibility conditions From  
0         0.0         6.0      Clear  BLR  
1         0.0         6.0      Clear  BLR  
2        10.0         4.0      Clear  BLR  


In [15]:
weather_all["Weather_Date"] = pd.to_datetime(
    weather_all["datetime"], errors="coerce"
).dt.date

print(weather_all[["From", "datetime", "Weather_Date"]].head())
print("Missing dates:", weather_all["Weather_Date"].isna().sum())

  From             datetime Weather_Date
0  BLR  2019-01-02T00:00:00   2019-01-02
1  BLR  2019-01-02T01:00:00   2019-01-02
2  BLR  2019-01-02T02:00:00   2019-01-02
3  BLR  2019-01-02T03:00:00   2019-01-02
4  BLR  2019-01-02T04:00:00   2019-01-02
Missing dates: 0


In [16]:
required = airport_days[["From", "Date"]].copy()
required["Date"] = pd.to_datetime(required["Date"]).dt.date

downloaded = weather_all[["From", "Weather_Date"]].copy()
downloaded = downloaded.rename(columns={"Weather_Date": "Date"})
downloaded = downloaded.drop_duplicates()

missing_days = required.merge(
    downloaded,
    on=["From", "Date"],
    how="left",
    indicator=True
)

missing_days = missing_days[
    missing_days["_merge"] == "left_only"
].drop(columns="_merge")

print("Airport-days remaining:", len(missing_days))
print(missing_days.groupby("From").size())

Airport-days remaining: 603
From
BLR    179
BOM    160
CCU    129
DEL    135
dtype: int64


In [ ]:
#SKIP

import pandas as pd
import requests
from io import StringIO
from pathlib import Path
import time

airports = {
    "BLR": "12.9716,77.5946",
    "BOM": "19.0896,72.8656",
    "CCU": "22.6547,88.4467",
    "DEL": "28.5562,77.1000"
}

start_date = pd.Timestamp("2019-01-02")
end_date = pd.Timestamp("2020-01-31")

elements = [
    "datetime",
    "windspeed",
    "precip",
    "humidity",
    "visibility",
    "sealevelpressure",
    "cloudcover",
    "conditions"
]

folder = Path("visual_crossing_weather")
folder.mkdir(exist_ok=True)

months = pd.period_range(start_date, end_date, freq="M")

for airport, location in airports.items():
    for month in months:

        start = max(month.start_time.normalize(), start_date)
        end = min(month.end_time.normalize(), end_date)

        filename = folder / f"{airport}_{month}.csv"

        # Skip files already downloaded
        if filename.exists():
            print(f"Skipping {airport} {month}")
            continue

        url = (
            "https://weather.visualcrossing.com/"
            "VisualCrossingWebServices/rest/services/timeline/"
            f"{location}/{start:%Y-%m-%d}/{end:%Y-%m-%d}"
        )

        params = {
            "key": API_KEY,
            "unitGroup": "metric",
            "include": "hours",
            "contentType": "csv",
            "elements": ",".join(elements)
        }

        response = requests.get(url, params=params, timeout=120)
        response.raise_for_status()

        chunk = pd.read_csv(StringIO(response.text))

        chunk["From"] = airport

        # Preserve the full local weather timestamp
        if "datetime" in chunk.columns:
            chunk["Weather_DateTime"] = pd.to_datetime(
                chunk["datetime"], errors="coerce"
            )

        chunk.to_csv(filename, index=False)

        print(f"Downloaded {airport} {month}: {len(chunk)} rows")

        time.sleep(0.3)

# Combine all downloaded files
files = sorted(folder.glob("*.csv"))

weather_all = pd.concat(
    [pd.read_csv(file) for file in files],
    ignore_index=True
)

weather_all.to_csv(
    "airport_weather_all.csv",
    index=False
)

print("\nDOWNLOAD COMPLETE")
print("Shape:", weather_all.shape)
print(weather_all["From"].value_counts())

Skipping BLR 2019-01
Skipping BLR 2019-02
Skipping BLR 2019-03
Skipping BLR 2019-04
Skipping BLR 2019-05
Skipping BLR 2019-06
Skipping BLR 2019-07
Skipping BLR 2019-08
Skipping BLR 2019-09
Skipping BLR 2019-10
Skipping BLR 2019-11
Skipping BLR 2019-12
Downloaded BLR 2020-01: 744 rows
Downloaded BOM 2019-01: 720 rows
Downloaded BOM 2019-02: 672 rows
Downloaded BOM 2019-03: 744 rows
Downloaded BOM 2019-04: 720 rows
Downloaded BOM 2019-05: 744 rows
Downloaded BOM 2019-06: 720 rows
Downloaded BOM 2019-07: 744 rows
Downloaded BOM 2019-08: 744 rows
Downloaded BOM 2019-09: 720 rows
Downloaded BOM 2019-10: 744 rows
Downloaded BOM 2019-11: 720 rows


HTTPError: 429 Client Error:  for url: https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/19.0896,72.8656/2019-12-01/2019-12-31?key=9MR7M2PVEJNEX3F9MXKF294PU&unitGroup=metric&include=hours&contentType=csv&elements=datetime%2Cwindspeed%2Cprecip%2Chumidity%2Cvisibility%2Csealevelpressure%2Ccloudcover%2Cconditions

In [17]:
import pandas as pd
import requests
import json
import time
from pathlib import Path

# Existing variables: df, airports, API_KEY
cache_dir = Path("weather_hour_cache")
cache_dir.mkdir(exist_ok=True)

needed = df[["From", "Scheduled_DateTime"]].copy()
needed["Hour"] = pd.to_datetime(
    needed["Scheduled_DateTime"]
).dt.floor("h")

needed = needed[["From", "Hour"]].dropna().drop_duplicates()

# Reuse any saved full-month downloads
monthly_dir = Path("visual_crossing_weather")
saved = []

for file in monthly_dir.glob("*.csv"):
    part = pd.read_csv(file)
    airport = file.stem.split("_")[0]
    month = file.stem.split("_")[1]

    # Handle CSVs where datetime contains only HH:MM:SS
    if "datetime" in part.columns:
        if part["datetime"].astype(str).str.contains(r"^\d{4}-").all():
            timestamps = pd.to_datetime(part["datetime"])
        else:
            # Monthly files contain consecutive hourly records
            start = pd.Timestamp(month + "-01")
            if month == "2019-01":
                start = pd.Timestamp("2019-01-02")
            timestamps = start + pd.to_timedelta(
                range(len(part)), unit="h"
            )

        part["Hour"] = timestamps
        part["From"] = airport
        saved.append(part)

if saved:
    existing = pd.concat(saved, ignore_index=True)
    existing = existing.drop_duplicates(["From", "Hour"])
else:
    existing = pd.DataFrame(columns=["From", "Hour"])

existing_keys = set(zip(existing["From"], existing["Hour"]))

missing = needed[
    ~needed.apply(
        lambda r: (r["From"], r["Hour"]) in existing_keys,
        axis=1
    )
]

print("Required hours:", len(needed))
print("Already saved:", len(needed) - len(missing))
print("Still needed:", len(missing))

session = requests.Session()

for i, row in enumerate(missing.itertuples(index=False), 1):
    airport = row.From
    hour = row.Hour

    filename = cache_dir / f"{airport}_{hour:%Y%m%d_%H}.json"

    if filename.exists():
        continue

    location = airports[airport]
    timestamp = hour.strftime("%Y-%m-%dT%H:%M:%S")

    url = (
        "https://weather.visualcrossing.com/"
        "VisualCrossingWebServices/rest/services/timeline/"
        f"{location}/{timestamp}"
    )

    params = {
        "key": API_KEY,
        "unitGroup": "metric",
        "include": "current",
        "contentType": "json"
    }

    response = session.get(url, params=params, timeout=60)

    if not response.ok:
        print(f"Stopped at {airport} {timestamp}")
        print(response.status_code, response.text[:200])
        break

    record = response.json().get("currentConditions", {})
    record["From"] = airport
    record["Hour"] = str(hour)

    filename.write_text(json.dumps(record))

    if i % 100 == 0:
        print(f"Downloaded {i} missing airport-hours")

    time.sleep(0.1)

print("Download run finished.") #RUN TMR, write in paper how going to use this as well to compare to literature, otherwise matches others.

Required hours: 4946
Already saved: 2008
Still needed: 2938
Downloaded 500 missing airport-hours
Downloaded 600 missing airport-hours
Downloaded 700 missing airport-hours
Downloaded 800 missing airport-hours
Downloaded 900 missing airport-hours
Downloaded 1000 missing airport-hours
Downloaded 1100 missing airport-hours
Downloaded 1200 missing airport-hours
Downloaded 1300 missing airport-hours
Downloaded 1400 missing airport-hours
Downloaded 1500 missing airport-hours
Downloaded 1600 missing airport-hours
Downloaded 1700 missing airport-hours
Downloaded 1800 missing airport-hours
Downloaded 1900 missing airport-hours
Downloaded 2000 missing airport-hours
Downloaded 2100 missing airport-hours
Downloaded 2200 missing airport-hours
Downloaded 2300 missing airport-hours
Downloaded 2400 missing airport-hours
Downloaded 2500 missing airport-hours
Downloaded 2600 missing airport-hours
Downloaded 2700 missing airport-hours
Downloaded 2800 missing airport-hours
Downloaded 2900 missing airport-h

In [19]:
import pandas as pd
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

epoch = 1546372800

print("UTC:", datetime.fromtimestamp(epoch, timezone.utc))
print("Mumbai:", datetime.fromtimestamp(epoch, ZoneInfo("Asia/Kolkata")))

UTC: 2019-01-01 20:00:00+00:00
Mumbai: 2019-01-02 01:30:00+05:30


In [20]:
import pandas as pd
import json
from pathlib import Path

df = pd.read_csv("flight_data_cleaned.csv")

# Create scheduled departure hour
df["Weather_Hour"] = pd.to_datetime(
    df["Scheduled_DateTime"]
).dt.floor("h")

# Load monthly weather files
weather_parts = []

for file in Path("visual_crossing_weather").glob("*.csv"):
    part = pd.read_csv(file)

    airport = file.stem.split("_")[0]
    month = file.stem.split("_")[1]

    if "datetime" in part.columns:
        if part["datetime"].astype(str).str.match(r"^\d{4}-").all():
            part["Weather_Hour"] = pd.to_datetime(part["datetime"])
        else:
            start = pd.Timestamp(month + "-01")
            if month == "2019-01":
                start = pd.Timestamp("2019-01-02")

            part["Weather_Hour"] = start + pd.to_timedelta(
                range(len(part)), unit="h"
            )

        part["From"] = airport
        weather_parts.append(part)

# Load cached JSON weather
for file in Path("weather_hour_cache").glob("*.json"):
    record = json.loads(file.read_text())

    # Use requested scheduled hour stored in filename
    parts = file.stem.split("_")
    airport = parts[0]
    hour = pd.to_datetime(parts[1] + parts[2], format="%Y%m%d%H")

    record["From"] = airport
    record["Weather_Hour"] = hour

    weather_parts.append(pd.DataFrame([record]))

weather = pd.concat(weather_parts, ignore_index=True)

# Prefer cached JSON records if both sources have the same airport/hour
weather = weather.drop_duplicates(
    subset=["From", "Weather_Hour"],
    keep="last"
)

C:\Users\iacom\AppData\Local\Temp\ipykernel_22188\695257893.py:50: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  weather = pd.concat(weather_parts, ignore_index=True)


In [21]:
# Remove old hourly weather features
old_weather_cols = [
    c for c in df.columns
    if c.startswith("weather__hourly__")
]

df = df.drop(columns=old_weather_cols)

# Select new Visual Crossing weather variables
weather_features = [
    "temp",
    "humidity",
    "precip",
    "windspeed",
    "cloudcover",
    "sealevelpressure",
    "visibility",
    "conditions"
]

available = [c for c in weather_features if c in weather.columns]

weather_new = weather[
    ["From", "Weather_Hour"] + available
].copy()

# Rename to distinguish new features
weather_new = weather_new.rename(
    columns={c: "VC_" + c for c in available}
)

# Match each flight to its scheduled departure hour
df = df.merge(
    weather_new,
    on=["From", "Weather_Hour"],
    how="left",
    validate="many_to_one"
)

print("Rows:", len(df))
print("Weather matching rate:",
      df["VC_temp"].notna().mean() * 100 if "VC_temp" in df else "No temperature column")

Rows: 10642
Weather matching rate: 48.966359706822026
